# Basic Factor Replication Demo

This notebook displays the first batch of reusable basic factors built from the local CRSP monthly panel and FF3 characteristics.

Each factor is winsorized by rebalance date at 1%/99%, split into 10 deciles, value-weighted within each decile, and evaluated as a D10-D01 spread unless noted by the signal definition.

In [ ]:
from pathlib import Path
import sys

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT.parent))
sys.path.insert(0, str(PROJECT / 'code'))

import pandas as pd
import plotly.graph_objects as go
from factors import FactorBacktester

RESULTS = PROJECT / 'results'
BASIC = RESULTS / 'basic_factors'
bt = FactorBacktester(periods_per_year=12)
cols = ['ann_return', 'ann_vol', 'sharpe', 'total_return', 'max_drawdown', 'hit_rate', 'avg_turnover', 'ann_turnover']
ic_cols = ['mean_ic', 'mean_rank_ic', 'mean_cos_ic', 'ic_ir', 'rank_ic_ir', 'cos_ic_ir', 'mean_r2']
BASIC


## Factor Spread Summary

In [ ]:
summary = pd.read_csv(BASIC / 'basic_factor_summary.csv', index_col=0)
summary[cols].style.format('{:.4f}')


## IC / R2 Summary

In [ ]:
ic_summary = pd.read_csv(BASIC / 'basic_factor_ic_r2_summary.csv', index_col=0)
ic_summary[ic_cols].style.format('{:.4f}')


## All Factor Spread Curves

In [ ]:
spread = pd.read_csv(BASIC / 'basic_factor_spread_returns.csv', parse_dates=['date'])
fig = go.Figure()
for name, g in spread.groupby('factor'):
    g = g.sort_values('date').copy()
    g['cum_return'] = (1 + g['ret']).cumprod() - 1
    fig.add_trace(go.Scatter(x=g['date'].to_numpy(dtype='datetime64[ns]'), y=g['cum_return'], name=name))
fig.update_yaxes(tickformat='.2%')
fig.update_layout(template='plotly_white', height=600, title='Basic factor spread cumulative returns')
fig.show()


## Inspect One Factor

Change `factor` to one of: `SIZE`, `BM`, `MOM_12_2`, `REV_1M`, `LOW_VOL_12M`, `ILLIQ`, `GROSS_PROFIT`, `OP_PROFIT`, `ROA`, `EARNINGS_YIELD`, `ASSET_GROWTH`, `SALES_GROWTH`, `LOW_LEVERAGE`, `CASH`, `LOW_ACCRUALS`, `LOW_CAPEX`.

In [ ]:
factor = 'MOM_12_2'
factor


In [ ]:
one = spread[spread['factor'].eq(factor)][['date', 'ret']]
fig = bt.plot_performance(one)
fig.update_layout(title=f'{factor} spread performance')
fig.show()


In [ ]:
layers = pd.read_csv(BASIC / 'basic_factor_layer_returns.csv', parse_dates=['date'])
one_layers = layers[layers['factor'].eq(factor)]
fig = bt.plot_layers(one_layers)
fig.update_layout(title=f'{factor} decile layer returns')
fig.show()


In [ ]:
ic = pd.read_csv(BASIC / 'basic_factor_ic.csv', parse_dates=['date'])
one_ic = ic[ic['factor'].eq(factor)]
fig = go.Figure()
for col in ['cum_ic', 'cum_rank_ic', 'cum_cos_ic']:
    fig.add_trace(go.Scatter(x=one_ic['date'].to_numpy(dtype='datetime64[ns]'), y=one_ic[col], name=col))
fig.update_layout(template='plotly_white', height=500, title=f'{factor} cumulative IC')
fig.show()
display(one_ic[['date', 'ic', 'rank_ic', 'cos_ic', 'n_assets']].tail())


## Definitions

In [ ]:
summary[['description']]
